In [ ]:
!pip install optuna
!pip install torchmetrics
!pip install flash-attn
!pip install scikit-optimize
!pip install pymfe

In [ ]:
import pywt
import os
import json
from types import SimpleNamespace
import torch
import torchvision
from torch import nn
import torch.nn.functional as F
from torchvision import transforms
from torch.utils.data import DataLoader

import time
import optuna
import random
import numpy as np
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from collections import Counter
from scipy.stats import gaussian_kde
from sklearn.metrics import f1_score
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from skopt import gp_minimize

from train import train, Parser
from bayes_opt.random_search import random_search, plot_param_interaction
from bayes_opt.bayes_opt import optuna_objective, plot_optuna_param_importance
from data_utils import load_denoising_n2n_train, load_denoising_test_mix

In [ ]:
%matplotlib inline

In [ ]:
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATASET_PATH = "../../dataset"
EPOCHS = 200
LR = 5e-4
BATCH_SIZE = 16
IMG_SIZE = 256
NOISE_LEVELS = [1, 2, 4, 8, 16]
MODEL_NAME = "rauden"
print(f"Using device: {DEVICE}")

TRIALS = 20

In [ ]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "max_split_size_mb:128"
torch.backends.cudnn.benchmark = True

In [ ]:
parser = Parser()
args = parser.parse()

args.epochs = 10
args.data_root = DATASET_PATH
args.exp_name = "bo_rauden"
args.debug = False

study = optuna.create_study(direction="maximize")
study.optimize(lambda trial: optuna_objective(trial, args), n_trials=30)

print("Best trial:")
print(study.best_trial.params)
print("Best score:", study.best_trial.value)

In [ ]:
df = study.trials_dataframe(attrs=("number", "value", "params"))
df = df.rename(columns={"value": "PSNR"})
df.columns = [c.replace("params_", "") for c in df.columns]
print(df.columns)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
import matplotlib.pyplot as plt

plot_feature_importance(df, target="PSNR")

In [ ]:
plot_param_interaction(
    df,
    x="lr",
    y="alpha",
    target="PSNR"
)
plot_hyperparams_3d(df, title="Optuna Hyperparameters vs PSNR")

In [ ]:
best = study.best_params
args.lr = best["lr"]
args.batch_size = best["batch_size"]
args.wd = best["wd"]
args.alpha = best["alpha"]
args.beta = best["beta"]
args.delta = best["delta"]
args.omega = best["omega"]
args.epochs = 100
train(args)

In [ ]:
transform = transforms.Compose([
    transforms.FiveCrop(IMG_SIZE),
    transforms.Lambda(lambda crops: torch.stack([
        fluore_to_tensor(crop) for crop in crops[:4]])),
    transforms.Lambda(lambda x: x.float().div(255).sub(0.5))
])

train_loader = load_denoising_n2n_train(
    DATASET_PATH,
    batch_size=BATCH_SIZE,
    noise_levels=NOISE_LEVELS,
    types=None,
    transform=transform,
    target_transform=transform,
    patch_size=IMG_SIZE
)

val_loader = load_denoising_test_mix(
    DATASET_PATH,
    batch_size=BATCH_SIZE,
    noise_levels=[1],
    transform=transform,
    patch_size=IMG_SIZE
)

In [ ]:
print("Meta-features shape:", meta_features.shape)
print("Meta-features names:")
for n, v in zip(feature_names, feature_values):
    print(f"{n}: {v:.4f}")

In [ ]:
values = np.array(feature_values)
names = np.array(feature_names)

mask = ~np.isnan(values)
values = values[mask]
names = names[mask]

plt.figure(figsize=(10, 12))
plt.barh(names, values)
plt.title("PyMFE Meta-features (RAUDen dataset)")
plt.xlabel("Value")
plt.tight_layout()
plt.show()